In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

LLM_MODEL = "distilbert-base-uncased"
data_submission_path = "submission.csv"

# Local path configurations
train_data_path = "kaggle/input/competitions/llm-classification-finetuning/train.csv"
test_data_path = "kaggle/input/competitions/llm-classification-finetuning/test.csv"
model_save_path = "./model"


# #Kaggle path configurations
# train_data_path = "/kaggle/input/competitions/llm-classification-finetuning/train.csv"
# test_data_path = "/kaggle/input/competitions/llm-classification-finetuning/test.csv"
# model_save_path = "/kaggle/input/models/yanxinye/llm-classifier-nicole-ye/other/default/1"


# To submit to Kaggle:
# 1. Manually upload the trained model to Kaggle's input -> Add model -> Upload model, and then copy the Kaggle path configurations above to load the model in the inference notebook.
# 2. Add the competition dataset to Kaggle's input, then use the Kaggle path configurations to load the data.
# 3. Turn off internet access in Kaggle's notebook settings to avoid any issues with loading the model or data.


In [3]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)

model.eval()

Using device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 22727.43it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [5]:
# 1) Load test data
test_df = pd.read_csv(test_data_path)

# 2) Prepare text in same format as training
test_df["text"] = (
    "Prompt:\n" + test_df["prompt"].astype(str) +
    "\n\nResponse A:\n" + test_df["response_a"].astype(str) +
    "\n\nResponse B:\n" + test_df["response_b"].astype(str)
)

test_dataset = Dataset.from_pandas(test_df[["text"]])

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



Map: 100%|██████████| 3/3 [00:00<00:00, 769.83 examples/s]


In [6]:
from transformers import DataCollatorWithPadding

# IMPORTANT: remove raw text before DataLoader
test_dataset = test_dataset.remove_columns(["text"])


data_collator = DataCollatorWithPadding(tokenizer)

from torch.utils.data import DataLoader

dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    collate_fn=data_collator   # ✅ THIS FIXES EVERYTHING
)

In [7]:
all_logits = []

with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits

        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)

In [8]:
print("nan logits?", torch.isnan(logits).any().item())
print("inf logits?", torch.isinf(logits).any().item())

probs = torch.softmax(logits, dim=-1).numpy()

out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv(data_submission_path, index=False)
print(out.head())

nan logits? False
inf logits? False
        id  winner_model_a  winner_model_b  winner_model_tie
0   136060        0.235536        0.265034          0.499429
1   211333        0.330544        0.389459          0.279997
2  1233961        0.374665        0.329594          0.295741
